In [6]:
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
github_token = secrets.get_secret("github_token")

github_username = "shaambhavi-dubey"
repo_name = "gnn-upi"
repo_url = f"https://{github_token}@github.com/{github_username}/{repo_name}.git"

!git clone {repo_url}
!cd gnn-upi && git config user.email "25bit087@sot.pdpu.ac.in"
!cd gnn-upi && git config user.name "shaambhavi-dubey"

!pip install torch_geometric --quiet

Cloning into 'gnn-upi'...
remote: Enumerating objects: 57, done.
remote: Counting objects: 100% (57/57), done.
remote: Compressing objects: 100% (46/46), done.
remote: Total 57 (delta 23), reused 30 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (57/57), 594.99 KiB | 4.47 MiB/s, done.
Resolving deltas: 100% (23/23), done.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 24.8 MB/s eta 0:00:00a 0:00:01


In [2]:
import pickle
import pandas as pd
import networkx as nx
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

base_path = "/kaggle/input/datasets/shaambhavidubey/gnn-synthetic-data/gnn-upi/data"

with open(f"{base_path}/synthetic_graph.pkl", "rb") as f:
    DirGr = pickle.load(f)

node_df = pd.read_csv(f"{base_path}/node_features.csv")
print(node_df.shape)

(10000, 5)


In [3]:
# we completely remove test nodes and edges from training unline gcn where we just lebelled and maksed them
# pick out the inductive test set
all_nodes = list(DirGr.nodes())
labels_dict = nx.get_node_attributes(DirGr, 'label')
labels_arr = np.array([labels_dict[n] for n in all_nodes])

from sklearn.model_selection import train_test_split

# first split: which nodes are even eligible to be seen during training vs held out entirely
t_nodes, te_nodes = train_test_split(
    all_nodes, test_size=0.15, stratify=labels_arr, random_state=42
)

print(f"Seen: {len(t_nodes)}")
print(f"Unseen : {len(te_nodes)}")
print(f"Unseen fraud count: {sum(labels_dict[n] for n in te_nodes)}")

Seen: 8500
Unseen : 1500
Unseen fraud count: 37


In [4]:
# build new graph w only trainung nodes and edges
train_graph = DirGr.subgraph(t_nodes).copy()
print(f"Training graph: {train_graph.number_of_nodes()} nodes, {train_graph.number_of_edges()} edges")

Training graph: 8500 nodes, 26421 edges


In [7]:
# build train validation from training graph, convert to pyg
from torch_geometric.utils import from_networkx

# attach the same features as gcn, computed on the training graph
for n in train_graph.nodes():
    incoming = [train_graph[u][n]['amount'] for u in train_graph.predecessors(n)]
    outgoing = [train_graph[n][t]['amount'] for t in train_graph.successors(n)]
    total = incoming + outgoing
    train_graph.nodes[n]['avg_amt'] = sum(total)/len(total) if total else 0.0
    train_graph.nodes[n]['max_amount'] = float(max(total)) if total else 0.0
    train_graph.nodes[n]['in_degree'] = train_graph.in_degree(n)
    train_graph.nodes[n]['out_degree'] = train_graph.out_degree(n)

train_data = from_networkx(train_graph, group_node_attrs=['account_age_days', 'avg_amt', 'max_amount', 'in_degree', 'out_degree'])

train_labels = torch.tensor([train_graph.nodes[n]['label'] for n in train_graph.nodes()], dtype=torch.long)
train_data.y = train_labels

print(train_data)

Data(edge_index=[2, 26421], label=[8500], amount=[26421], x=[8500, 5], y=[8500])


In [8]:
# now split train_graph's nodes into actual train/val (no separate test here test is the te_nodes we already set aside)
n_train_nodes = train_data.num_nodes
train_indices = np.arange(n_train_nodes)

tr_idx, val_idx = train_test_split(
    train_indices, test_size=0.15, stratify=train_data.y.numpy(), random_state=42
)

train_mask = torch.zeros(n_train_nodes, dtype=torch.bool)
val_mask = torch.zeros(n_train_nodes, dtype=torch.bool)
train_mask[tr_idx] = True
val_mask[val_idx] = True

train_data.train_mask = train_mask
train_data.val_mask = val_mask

print(f"Train: {train_mask.sum()}, Val: {val_mask.sum()}")
print(f"Train fraud: {train_data.y[train_mask].sum()}, Val fraud: {train_data.y[val_mask].sum()}")

Train: 7225, Val: 1275
Train fraud: 181, Val fraud: 32


In [9]:
# normalization
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
x_scaled = scaler.fit_transform(train_data.x.numpy())
train_data.x = torch.tensor(x_scaled, dtype=torch.float)

In [10]:
# graphsage model
from torch_geometric.nn import SAGEConv

class GraphSAGE(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.conv2(x, edge_index)
        return x

In [11]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
train_data = train_data.to(device)

model = GraphSAGE(in_channels=train_data.x.shape[1], hidden_channels=64, out_channels=2).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)

class_counts = torch.bincount(train_data.y[train_data.train_mask])
raw_ratio = (class_counts.sum() / class_counts).float()
class_weights = torch.sqrt(raw_ratio).to(device)  # softened, same fix as notebook 04
criterion = nn.CrossEntropyLoss(weight=class_weights)

print(f"Class weights: {class_weights}")

Class weights: tensor([1.0128, 6.3180])


In [12]:
from sklearn.metrics import average_precision_score

def train():
    model.train()
    optimizer.zero_grad()
    out = model(train_data.x, train_data.edge_index)
    loss = criterion(out[train_data.train_mask], train_data.y[train_data.train_mask])
    loss.backward()
    optimizer.step()
    return loss.item()

def eval_val():
    model.eval()
    with torch.no_grad():
        out = model(train_data.x, train_data.edge_index)
        probs = F.softmax(out, dim=1)[:, 1]
        val_probs = probs[train_data.val_mask].cpu().numpy()
        val_labels = train_data.y[train_data.val_mask].cpu().numpy()
        return average_precision_score(val_labels, val_probs)

for epoch in range(400):
    loss = train()
    if epoch % 40 == 0:
        val_pr_auc = eval_val()
        print(f"Epoch {epoch}, Loss: {loss:.4f}, Val PR-AUC: {val_pr_auc:.3f}")

Epoch 0, Loss: 0.7364, Val PR-AUC: 0.022
Epoch 40, Loss: 0.0386, Val PR-AUC: 0.978
Epoch 80, Loss: 0.0191, Val PR-AUC: 0.989
Epoch 120, Loss: 0.0122, Val PR-AUC: 0.993
Epoch 160, Loss: 0.0075, Val PR-AUC: 0.998
Epoch 200, Loss: 0.0094, Val PR-AUC: 0.992
Epoch 240, Loss: 0.0092, Val PR-AUC: 0.998
Epoch 280, Loss: 0.0060, Val PR-AUC: 0.995
Epoch 320, Loss: 0.0070, Val PR-AUC: 1.000
Epoch 360, Loss: 0.0065, Val PR-AUC: 1.000


In [14]:
# build the graph again to test now
for n in DirGr.nodes():
    incoming = [DirGr[u][n]['amount'] for u in DirGr.predecessors(n)]
    outgoing = [DirGr[n][t]['amount'] for t in DirGr.successors(n)]
    total = incoming + outgoing
    DirGr.nodes[n]['avg_amt'] = sum(total)/len(total) if total else 0.0
    DirGr.nodes[n]['max_amount'] = float(max(total)) if total else 0.0
    DirGr.nodes[n]['in_degree'] = DirGr.in_degree(n)
    DirGr.nodes[n]['out_degree'] = DirGr.out_degree(n)

full_data = from_networkx(DirGr, group_node_attrs=['account_age_days', 'avg_amt', 'max_amount', 'in_degree', 'out_degree'])
full_labels = torch.tensor([DirGr.nodes[n]['label'] for n in DirGr.nodes()], dtype=torch.long)
full_data.y = full_labels

# use the scaler fitted on train_graph, using the 8500 nodes already fir not doing fit again because it will learn the unseen ones as well
x_scaled_full = scaler.transform(full_data.x.numpy())
full_data.x = torch.tensor(x_scaled_full, dtype=torch.float)
full_data = full_data.to(device)

# build a mask marking which nodes are the truly unseen ones
node_list = list(DirGr.nodes())
unseen_set = set(te_nodes)
inductive_test_mask = torch.tensor([n in unseen_set for n in node_list], dtype=torch.bool).to(device)

print(f"Inductive test nodes: {inductive_test_mask.sum().item()}")

Inductive test nodes: 1500


In [15]:
model.eval()
with torch.no_grad():
    out = model(full_data.x, full_data.edge_index)
    probs = F.softmax(out, dim=1)[:, 1]
    preds = out.argmax(dim=1)

inductive_preds = preds[inductive_test_mask].cpu().numpy()
inductive_labels = full_data.y[inductive_test_mask].cpu().numpy()
inductive_probs = probs[inductive_test_mask].cpu().numpy()

from sklearn.metrics import classification_report, average_precision_score, f1_score

print(classification_report(inductive_labels, inductive_preds))
print(f"PR-AUC (inductive, unseen nodes): {average_precision_score(inductive_labels, inductive_probs):.3f}")
print(f"F1 (inductive, unseen nodes): {f1_score(inductive_labels, inductive_preds):.3f}")

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1463
           1       0.97      1.00      0.99        37

    accuracy                           1.00      1500
   macro avg       0.99      1.00      0.99      1500
weighted avg       1.00      1.00      1.00      1500

PR-AUC (inductive, unseen nodes): 0.999
F1 (inductive, unseen nodes): 0.987


In [16]:
results_05 = {
    "notebook": "05_graphsage",
    "architecture": "2-layer SAGEConv, hidden=64, dropout=0.5",
    "features_used": ["account_age_days", "avg_amt", "max_amount", "in_degree", "out_degree"],
    "class_weighting": "sqrt(imbalance_ratio)",
    "inductive_setup": "1500 nodes (~15%) fully removed from graph during training, reintroduced only at test time",
    "val_pr_auc_seen_nodes": 1.000,
    "inductive_test_precision_fraud": 0.97,
    "inductive_test_recall_fraud": 1.00,
    "inductive_test_f1": 0.987,
    "inductive_test_pr_auc": 0.999
}

import json
with open("gnn-upi/data/results_05_graphsage.json", "w") as f:
    json.dump(results_05, f, indent=2)

print("saved")

saved


In [19]:
from IPython.display import FileLink
FileLink("gnn-upi/data/results_05_graphsage.json")

/kaggle/working/gnn-upi/data/results_05_graphsage.json